# Enterprise Knowledge Navigator
## DAY 4: Streamlit Frontend + RAGAS Evaluation + GitHub
### Final day - complete the project!

**What we build today:**
- Master restart cell: recover from any disconnection instantly
- Streamlit chat UI: professional frontend with source highlighting
- RAGAS evaluation: measure faithfulness, relevancy, context scores
- Live evaluation dashboard inside Streamlit
- GitHub README and requirements.txt
- Final project complete!


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## MASTER RESTART CELL
Run this first every time. Installs everything and reloads all data.


In [2]:
# ============================================================
# MASTER RESTART CELL - run this first after any disconnection
# ============================================================
!pip install -q sentence-transformers chromadb rank-bm25 groq pymupdf
!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pip install -q streamlit
!pip install -q ragas datasets

import os, re, time, pickle, json, hashlib, torch, threading
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
import chromadb
from rank_bm25 import BM25Okapi
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()

BASE = "/content/drive/MyDrive/EnterpriseKnowledgeNavigator"
device = "cuda" if torch.cuda.is_available() else "cpu"

def tokenize_bm25(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 -]", " ", text)
    return [t for t in text.split() if len(t) > 1]

def embed_text(text):
    return embedding_model.encode(text, normalize_embeddings=True)

def call_groq(prompt, system="You are a helpful Tesla company expert.", max_tokens=500):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.1
    )
    return response.choices[0].message.content

def vector_search(query, n=10):
    qe = embed_text(query).tolist()
    results = collection.query(
        query_embeddings=[qe], n_results=n,
        include=["documents", "metadatas", "distances"]
    )
    return [{"text": d, "metadata": m, "score": round(1-dist, 4)}
            for d, m, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])]

def bm25_search(query, n=10):
    tokens = tokenize_bm25(query)
    scores = bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    return [{"text": all_chunks[i]["text"], "metadata": all_chunks[i]["metadata"], "score": round(float(scores[i]), 4)}
            for i in top_idx if scores[i] > 0]

def hybrid_search(query, n_final=10, n_retrieve=20):
    vec = vector_search(query, n=n_retrieve)
    bm = bm25_search(query, n=n_retrieve)
    rrf, data = {}, {}
    for rank, r in enumerate(vec):
        cid = r["metadata"]["chunk_id"]
        rrf[cid] = rrf.get(cid, 0) + 1.0/(rank+60)
        data[cid] = r
    for rank, r in enumerate(bm):
        cid = r["metadata"]["chunk_id"]
        rrf[cid] = rrf.get(cid, 0) + 1.0/(rank+60)
        data[cid] = r
    sorted_ids = sorted(rrf.keys(), key=lambda k: rrf[k], reverse=True)
    results = []
    for cid in sorted_ids[:n_final]:
        item = dict(data[cid])
        item["rrf_score"] = round(rrf[cid], 6)
        results.append(item)
    return results

def rerank_chunks(query, chunks, top_n=5):
    if not chunks: return []
    scores = reranker.predict([(query, c["text"]) for c in chunks])
    for i, c in enumerate(chunks):
        c["rerank_score"] = round(float(scores[i]), 4)
    return sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)[:top_n]

print("Loading models...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
chroma_client = chromadb.PersistentClient(path=BASE + "/chromadb")
collection = chroma_client.get_collection("tesla_knowledge_base")
with open(BASE + "/bm25_index/bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)
bm25 = bm25_data["index"]
all_chunks = bm25_data["chunks"]

print("=" * 50)
print("MASTER RESTART COMPLETE!")
print("Device: " + device)
print("Vectors: " + str(collection.count()))
print("Chunks: " + str(len(all_chunks)))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

MASTER RESTART COMPLETE!
Device: cuda
Vectors: 543
Chunks: 543


## STEP 2: Groq API Key


In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API"
groq_client = Groq(api_key=GROQ_API_KEY)
test = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say: Ready!"}],
    max_tokens=10
)
print("Groq: " + test.choices[0].message.content)


Groq: Ready!


## STEP 3: ConversationalRAG and FastAPI Setup

Redefine the ConversationalRAG class and FastAPI app from Day 3.
Then run the server so Streamlit can connect to it.


In [4]:
class ConversationalRAG:
    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self.history = []
        self.summary = ""
        self.eval_log = []

    def compress(self):
        old = self.history[:5]
        self.history = self.history[5:]
        turns = "".join(["Q: " + t["question"] + "\nA: " + t["answer"] + "\n\n" for t in old])
        self.summary = (self.summary + " " + call_groq(
            "Summarize in 3 sentences keeping key facts:\n\n" + turns, max_tokens=150
        )).strip()

    def chat(self, question):
        if len(self.history) >= self.max_turns:
            self.compress()
        candidates = hybrid_search(question, n_final=20)
        top_chunks = rerank_chunks(question, candidates, top_n=4)
        context_parts, sources = [], []
        for i, chunk in enumerate(top_chunks):
            label = "[Source " + str(i+1) + ": " + chunk["metadata"]["source"] + " p" + str(chunk["metadata"]["page_number"]) + "]"
            context_parts.append(label + "\n" + chunk["text"])
            sources.append({"label": label, "source": chunk["metadata"]["source"],
                           "page": chunk["metadata"]["page_number"],
                           "category": chunk["metadata"].get("category", "general")})
        context = "\n\n".join(context_parts)
        history_parts = []
        if self.summary:
            history_parts.append("Summary: " + self.summary)
        for t in self.history[-5:]:
            history_parts.append("User: " + t["question"])
            history_parts.append("Assistant: " + t["answer"])
        prompt = "You are a Tesla expert. Cite sources like [Source 1].\n\n"
        if history_parts:
            prompt += "History:\n" + "\n".join(history_parts) + "\n\n"
        prompt += "Context:\n" + context + "\n\nQuestion: " + question + "\n\nAnswer:"
        answer = call_groq(prompt, max_tokens=350)
        self.history.append({"question": question, "answer": answer})
        self.eval_log.append({"question": question, "answer": answer,
                              "contexts": [c["text"] for c in top_chunks]})
        return {"answer": answer, "sources": sources, "turn": len(self.history),
                "contexts": [c["text"] for c in top_chunks]}


chat_session = ConversationalRAG(max_turns=10)
print("ConversationalRAG ready!")


ConversationalRAG ready!


In [5]:
class ChatRequest(BaseModel):
    question: str
    use_graph: bool = True

app = FastAPI(title="Enterprise Knowledge Navigator API")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "healthy", "vectors": collection.count(), "chunks": len(all_chunks)}

@app.get("/stats")
def stats():
    return {
        "total_chunks": len(all_chunks),
        "total_vectors": collection.count(),
        "total_documents": len(set(c["metadata"]["source"] for c in all_chunks)),
        "embedding_model": "all-MiniLM-L6-v2",
        "llm": "Groq Llama 3 70B"
    }

@app.post("/chat")
def chat_endpoint(request: ChatRequest):
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")
    try:
        response = chat_session.chat(request.question)
        return {"answer": response["answer"], "sources": response["sources"],
                "turn": response["turn"], "query": request.question,
                "contexts": response["contexts"]}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/chat/reset")
def reset():
    global chat_session
    chat_session = ConversationalRAG(max_turns=10)
    return {"message": "Conversation reset"}

@app.get("/eval/log")
def get_eval_log():
    return {"log": chat_session.eval_log}

# Start server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

import requests as req
try:
    existing = req.get("http://localhost:8000/health", timeout=2)
    print("FastAPI already running! Health: " + str(existing.json()))
except:
    t = threading.Thread(target=run_server, daemon=True)
    t.start()
    time.sleep(3)
    health_check = req.get("http://localhost:8000/health")
    print("FastAPI started! Health: " + str(health_check.json()))

API_URL = "http://localhost:8000"
print("API URL: " + API_URL)


FastAPI started! Health: {'status': 'healthy', 'vectors': 543, 'chunks': 543}
API URL: http://localhost:8000


## STEP 4: RAGAS Evaluation

**What is RAGAS?**
RAGAS = Retrieval Augmented Generation Assessment.
It automatically measures how good your RAG system is.

**Three key metrics:**

Faithfulness: Is the answer supported by the retrieved context?
Score 1.0 = every claim in the answer comes from the context.
Score 0.0 = answer is hallucinated with no support from context.

Answer Relevancy: Does the answer actually address the question?
Score 1.0 = answer directly and completely answers the question.
Score 0.0 = answer is off-topic or incomplete.

Context Precision: Are the retrieved chunks actually relevant?
Score 1.0 = all retrieved chunks were useful for answering.
Score 0.0 = retrieved chunks were irrelevant noise.

Good RAG systems score above 0.7 on all three metrics.


In [6]:
# Run evaluation questions first to build eval log
print("Running evaluation questions to build eval log...")
print("This populates chat_session.eval_log with question-answer-context triples")
print()

eval_questions = [
    "What is Tesla revenue in 2023?",
    "How many vehicles did Tesla deliver in 2023?",
    "What is the Tesla 4680 battery cell?",
    "Where are Tesla Gigafactories located?",
    "What is Tesla Autopilot?",
    "What products does Tesla Energy sell?",
    "Who founded Tesla?",
    "What is the Tesla Supercharger network?",
    "What is Tesla Model Y?",
    "What is Tesla FSD software?",
]

eval_results = []
for q in eval_questions:
    print("Evaluating: " + q[:60])
    resp = chat_session.chat(q)
    eval_results.append({
        "question": q,
        "answer": resp["answer"],
        "contexts": resp["contexts"]
    })
    time.sleep(0.5)

print("\nEval questions answered: " + str(len(eval_results)))


Running evaluation questions to build eval log...
This populates chat_session.eval_log with question-answer-context triples

Evaluating: What is Tesla revenue in 2023?
Evaluating: How many vehicles did Tesla deliver in 2023?
Evaluating: What is the Tesla 4680 battery cell?
Evaluating: Where are Tesla Gigafactories located?
Evaluating: What is Tesla Autopilot?
Evaluating: What products does Tesla Energy sell?
Evaluating: Who founded Tesla?
Evaluating: What is the Tesla Supercharger network?
Evaluating: What is Tesla Model Y?
Evaluating: What is Tesla FSD software?

Eval questions answered: 10


In [7]:
# Compute RAGAS scores using Groq as judge
# We implement lightweight RAGAS-style scoring since full RAGAS needs OpenAI

def score_faithfulness(question, answer, contexts):
    """
    Faithfulness: Is every claim in the answer supported by the context?
    LLM-as-judge approach: ask Groq to score 0 to 1.
    """
    context_text = "\n".join(contexts[:3])
    prompt = (
        "Score the faithfulness of this answer to the context on a scale of 0.0 to 1.0.\n"
        "Faithfulness means every claim in the answer is supported by the context.\n"
        "Return ONLY a number between 0.0 and 1.0, nothing else.\n\n"
        "Context: " + context_text[:800] + "\n\n"
        "Answer: " + answer[:400] + "\n\n"
        "Score:"
    )
    try:
        result = call_groq(prompt, max_tokens=5)
        return round(min(1.0, max(0.0, float(result.strip()))), 3)
    except:
        return 0.5

def score_relevancy(question, answer):
    """
    Answer Relevancy: Does the answer address the question?
    """
    prompt = (
        "Score how well this answer addresses the question on a scale of 0.0 to 1.0.\n"
        "Return ONLY a number between 0.0 and 1.0, nothing else.\n\n"
        "Question: " + question + "\n"
        "Answer: " + answer[:400] + "\n\n"
        "Score:"
    )
    try:
        result = call_groq(prompt, max_tokens=5)
        return round(min(1.0, max(0.0, float(result.strip()))), 3)
    except:
        return 0.5

def score_context_precision(question, contexts):
    """
    Context Precision: Are the retrieved contexts relevant to the question?
    """
    context_text = "\n".join(contexts[:3])
    prompt = (
        "Score how relevant these retrieved contexts are to the question on a scale of 0.0 to 1.0.\n"
        "Return ONLY a number between 0.0 and 1.0, nothing else.\n\n"
        "Question: " + question + "\n"
        "Contexts: " + context_text[:800] + "\n\n"
        "Score:"
    )
    try:
        result = call_groq(prompt, max_tokens=5)
        return round(min(1.0, max(0.0, float(result.strip()))), 3)
    except:
        return 0.5


print("Computing RAGAS scores for " + str(len(eval_results)) + " questions...")
print("This takes 2-3 minutes...")

scored_results = []
for item in tqdm(eval_results):
    f = score_faithfulness(item["question"], item["answer"], item["contexts"])
    r = score_relevancy(item["question"], item["answer"])
    c = score_context_precision(item["question"], item["contexts"])
    scored_results.append({
        "question": item["question"],
        "faithfulness": f,
        "answer_relevancy": r,
        "context_precision": c,
        "average": round((f + r + c) / 3, 3)
    })
    time.sleep(0.3)

avg_faith = round(sum(r["faithfulness"] for r in scored_results) / len(scored_results), 3)
avg_rel = round(sum(r["answer_relevancy"] for r in scored_results) / len(scored_results), 3)
avg_ctx = round(sum(r["context_precision"] for r in scored_results) / len(scored_results), 3)
avg_overall = round((avg_faith + avg_rel + avg_ctx) / 3, 3)

ragas_summary = {
    "faithfulness": avg_faith,
    "answer_relevancy": avg_rel,
    "context_precision": avg_ctx,
    "overall": avg_overall,
    "questions_evaluated": len(scored_results),
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "detailed_results": scored_results
}

with open(BASE + "/outputs/eval/ragas_scores.json", "w") as f:
    json.dump(ragas_summary, f, indent=2)

print("\n" + "=" * 50)
print("RAGAS EVALUATION RESULTS")
print("=" * 50)
print("Faithfulness:      " + str(avg_faith) + " / 1.0")
print("Answer Relevancy:  " + str(avg_rel) + " / 1.0")
print("Context Precision: " + str(avg_ctx) + " / 1.0")
print("Overall Score:     " + str(avg_overall) + " / 1.0")
print("=" * 50)
print("Saved to Drive!")


Computing RAGAS scores for 10 questions...
This takes 2-3 minutes...


100%|██████████| 10/10 [00:39<00:00,  3.93s/it]



RAGAS EVALUATION RESULTS
Faithfulness:      0.68 / 1.0
Answer Relevancy:  0.91 / 1.0
Context Precision: 0.92 / 1.0
Overall Score:     0.837 / 1.0
Saved to Drive!


## STEP 5: Build Streamlit Frontend

**What is Streamlit?**
Streamlit converts Python scripts into web applications.
No HTML, CSS, or JavaScript needed.
We write Python and get a professional chat interface.

Our Streamlit app has:
- Chat interface for asking questions
- Source citations shown below each answer
- Conversation history displayed
- RAGAS evaluation dashboard in sidebar
- Knowledge base statistics

We write the app to a file then run it in the next cell.


In [9]:
streamlit_code = '''import streamlit as st
import requests
import json

API_URL = "http://localhost:8000"

st.set_page_config(page_title="Enterprise Knowledge Navigator", page_icon="🏢", layout="wide")

css = """
<style>
.main-title { font-size: 2rem; font-weight: bold; color: #1f4e79; }
.subtitle { font-size: 1rem; color: #666; margin-bottom: 1rem; }
.source-box { background: #f0f7ff; border-left: 3px solid #1f4e79; padding: 8px; margin: 4px 0; border-radius: 4px; font-size: 0.85rem; }
.metric-box { background: #e8f5e9; padding: 10px; border-radius: 8px; text-align: center; }
.score-good { color: #2e7d32; font-weight: bold; font-size: 1.5rem; }
.score-mid  { color: #f57c00; font-weight: bold; font-size: 1.5rem; }
</style>
"""
st.markdown(css, unsafe_allow_html=True)
st.markdown('<div class="main-title">Enterprise Knowledge Navigator</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">Advanced RAG | Tesla Knowledge Base | Groq Llama 3</div>', unsafe_allow_html=True)
st.divider()

with st.sidebar:
    st.header("Knowledge Base Stats")
    try:
        stats = requests.get(API_URL + "/stats", timeout=5).json()
        st.metric("Total Chunks", stats["total_chunks"])
        st.metric("Total Vectors", stats["total_vectors"])
        st.metric("Documents", stats["total_documents"])
        st.caption("Model: " + stats["embedding_model"])
        st.caption("LLM: " + stats["llm"])
    except:
        st.error("API not reachable")

    st.divider()
    st.header("RAGAS Scores")
    try:
        ragas_path = "/content/drive/MyDrive/EnterpriseKnowledgeNavigator/outputs/eval/ragas_scores.json"
        with open(ragas_path) as f:
            ragas = json.load(f)
        def score_color(s):
            return "score-good" if s >= 0.7 else "score-mid"
        for metric, key in [("Faithfulness","faithfulness"),("Answer Relevancy","answer_relevancy"),("Context Precision","context_precision"),("Overall","overall")]:
            val = ragas[key]
            st.markdown('<div class="metric-box"><div>' + metric + '</div><div class="' + score_color(val) + '">' + str(val) + '</div></div>', unsafe_allow_html=True)
            st.markdown("<br>", unsafe_allow_html=True)
        st.caption("Evaluated on " + str(ragas["questions_evaluated"]) + " questions")
    except:
        st.info("Run RAGAS evaluation cell first")

    st.divider()
    if st.button("Reset Conversation"):
        requests.post(API_URL + "/chat/reset")
        st.session_state.messages = []
        st.session_state.sources_log = []
        st.rerun()

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Chat with Tesla Knowledge Base")
    if "messages" not in st.session_state:
        st.session_state.messages = []
    if "sources_log" not in st.session_state:
        st.session_state.sources_log = []
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])
    if prompt := st.chat_input("Ask about Tesla..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.write(prompt)
        with st.chat_message("assistant"):
            with st.spinner("Searching knowledge base..."):
                try:
                    resp = requests.post(API_URL + "/chat", json={"question": prompt}, timeout=60).json()
                    answer = resp["answer"]
                    sources = resp["sources"]
                    turn = resp["turn"]
                    st.write(answer)
                    st.session_state.messages.append({"role": "assistant", "content": answer})
                    st.session_state.sources_log.append({"question": prompt, "sources": sources})
                    st.caption("Turn " + str(turn) + "/10")
                except Exception as e:
                    st.error("Error: " + str(e))

with col2:
    st.subheader("Sources")
    if st.session_state.sources_log:
        latest = st.session_state.sources_log[-1]
        st.caption("Latest answer sources:")
        for s in latest["sources"]:
            st.markdown('<div class="source-box">File: ' + s["source"] + "<br>Page: " + str(s["page"]) + " | " + s.get("category","") + "</div>", unsafe_allow_html=True)
    else:
        st.info("Ask a question to see sources")

st.divider()
st.subheader("Try These Questions")
sample_qs = ["What is Tesla revenue in 2023?","How does Tesla Autopilot work?","What is the 4680 battery cell?","How many Gigafactories does Tesla have?","What is Tesla Supercharger network?"]
cols = st.columns(len(sample_qs))
for i, q in enumerate(sample_qs):
    with cols[i]:
        if st.button(q[:28] + "..", key="sq" + str(i)):
            st.session_state.messages.append({"role": "user", "content": q})
            try:
                resp = requests.post(API_URL + "/chat", json={"question": q}, timeout=60).json()
                st.session_state.messages.append({"role": "assistant", "content": resp["answer"]})
                st.session_state.sources_log.append({"question": q, "sources": resp["sources"]})
            except:
                pass
            st.rerun()
'''

with open("/content/app.py", "w") as f:
    f.write(streamlit_code)

BASE = "/content/drive/MyDrive/EnterpriseKnowledgeNavigator"
with open(BASE + "/notebooks/streamlit_app.py", "w") as f:
    f.write(streamlit_code)

print("Streamlit app written successfully!")
print("Now run Step 6 to launch it.")

Streamlit app written successfully!
Now run Step 6 to launch it.


In [14]:
!pkill -f streamlit
!pkill -f localtunnel

In [15]:
!pip install pyngrok

In [17]:
from pyngrok import ngrok

ngrok.set_auth_token("3A6MaODbh98O9qOteGzTzfjXpEs_B4ZC1DxgK12B2zqZsGLE")

In [20]:
!pkill -f streamlit
!pkill -f ngrok

## STEP 6: Run Streamlit App

We use localtunnel to get a public URL for Streamlit.
Open the URL that appears below in a new browser tab.
Take a screenshot of the running app for your GitHub submission.


In [18]:
from pyngrok import ngrok
import subprocess
import time

# Kill old tunnels
ngrok.kill()

# Start Streamlit
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"]
)

time.sleep(8)

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print("Streamlit is running!")
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://sportier-samual-unhateful.ngrok-free.dev" -> "http://localhost:8501"


## STEP 7: Create GitHub README

The README is the most important file for your GitHub submission.
Evaluators read this first. A good README shows professionalism.


In [22]:
readme_content = """# Enterprise Knowledge Navigator
## Milestone Project 2 - AlgoProfessor AI R&D Internship 2026

> Build an AI knowledge-worker using advanced RAG to become an expert on all company-related matters.

## Architecture

```
User Query
    |
    v
HyDE + Multi-Query Generation (Groq Llama 3)
    |
    v
Hybrid Search: BM25 + Vector (RRF Fusion)
    |
    v
Graph RAG Enhancement (Neo4j entity relationships)
    |
    v
Cross-Encoder Reranking (ms-marco-MiniLM)
    |
    v
Groq Llama 3 70B Answer Generation
    |
    v
Cited Answer with Page Numbers
```

## Features

- 50+ Tesla company documents ingested (Wikipedia + reports + structured data)
- Graph RAG with Neo4j knowledge graph capturing entity relationships
- HyDE retrieval: hypothetical document embeddings for better search
- Multi-query: 3 query variants merged for higher recall
- Hybrid Search: BM25 + Vector combined with RRF score fusion
- Cross-encoder reranking for precise top-5 selection
- Conversational memory: 10-turn window with LLM summary compression
- FastAPI backend with REST endpoints
- Streamlit frontend with source highlighting
- Live RAGAS evaluation dashboard

## Tech Stack

| Component | Technology | Cost |
|-----------|-----------|------|
| LLM | Groq Llama 3 70B | Free |
| Embeddings | sentence-transformers all-MiniLM-L6-v2 | Free |
| Vector DB | ChromaDB (persistent) | Free |
| Keyword Search | BM25Okapi | Free |
| Reranker | cross-encoder ms-marco-MiniLM-L-6-v2 | Free |
| Graph DB | Neo4j AuraDB | Free tier |
| Backend | FastAPI | Free |
| Frontend | Streamlit | Free |
| Evaluation | RAGAS (LLM-as-judge) | Free |

## RAGAS Evaluation Results

| Metric | Score |
|--------|-------|
| Faithfulness | See outputs/eval/ragas_scores.json |
| Answer Relevancy | See outputs/eval/ragas_scores.json |
| Context Precision | See outputs/eval/ragas_scores.json |

## Project Structure

```
day15-enterprise-knowledge-navigator/
├── notebooks/
│   ├── Day1_Ingestion.ipynb
│   ├── Day2_RAG_Pipeline.ipynb
│   ├── Day3_GraphRAG_FastAPI.ipynb
│   └── Day4_Frontend_RAGAS.ipynb
├── outputs/
│   ├── eval/ragas_scores.json
│   ├── day1_summary.json
│   ├── day2_summary.json
│   ├── day3_summary.json
│   └── screenshots/
├── requirements.txt
└── README.md
```

## How to Run

1. Open any Day notebook in Google Colab
2. Run the MASTER RESTART CELL first
3. Add your Groq API key
4. Run all cells in order

## Dataset

Tesla public documents collected from:
- Wikipedia (25 pages: products, technology, manufacturing, leadership)
- Structured company documents (financial, HR, technology overview)
- Additional EV and technology context documents

## Author

AlgoProfessor AI R&D Internship 2026 - Milestone Project 2
"""

requirements = """sentence-transformers
chromadb
rank-bm25
groq
pymupdf
pdfplumber
neo4j
fastapi
uvicorn
streamlit
pyngrok
nest-asyncio
requests
beautifulsoup4
lxml
tqdm
numpy
datasets
"""

with open(BASE + "/notebooks/README.md", "w") as f:
    f.write(readme_content)
print("README.md written!")

with open(BASE + "/notebooks/requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt written!")

print("\nDownload these files from Drive:")
print("  " + BASE + "/notebooks/README.md")
print("  " + BASE + "/notebooks/requirements.txt")


README.md written!
requirements.txt written!

Download these files from Drive:
  /content/drive/MyDrive/EnterpriseKnowledgeNavigator/notebooks/README.md
  /content/drive/MyDrive/EnterpriseKnowledgeNavigator/notebooks/requirements.txt


In [23]:
summary_d4 = {
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "components_built": [
        "streamlit_chat_frontend",
        "ragas_evaluation_dashboard",
        "faithfulness_scoring",
        "answer_relevancy_scoring",
        "context_precision_scoring",
        "github_readme",
        "requirements_txt"
    ],
    "status": "PROJECT COMPLETE"
}

with open(BASE + "/outputs/day4_summary.json", "w") as f:
    json.dump(summary_d4, f, indent=2)

print("=" * 60)
print("PROJECT COMPLETE!")
print("=" * 60)
print()
print("GITHUB SUBMISSION CHECKLIST:")
print("[ ] Day1_Ingestion.ipynb uploaded")
print("[ ] Day2_RAG_Pipeline.ipynb uploaded")
print("[ ] Day3_GraphRAG_FastAPI.ipynb uploaded")
print("[ ] Day4_Frontend_RAGAS.ipynb uploaded")
print("[ ] README.md added")
print("[ ] requirements.txt added")
print("[ ] outputs/eval/ragas_scores.json added")
print("[ ] Screenshot of Streamlit UI added")
print()
print("COMMIT MESSAGE:")
print("[Day-15] Enterprise Knowledge Navigator - Complete Advanced RAG System")
print()
print("EMAIL TO: ceo@algoprofessor.com")
print("Subject: Day 15 Milestone - Enterprise Knowledge Navigator Submitted")


PROJECT COMPLETE!

GITHUB SUBMISSION CHECKLIST:
[ ] Day1_Ingestion.ipynb uploaded
[ ] Day2_RAG_Pipeline.ipynb uploaded
[ ] Day3_GraphRAG_FastAPI.ipynb uploaded
[ ] Day4_Frontend_RAGAS.ipynb uploaded
[ ] README.md added
[ ] requirements.txt added
[ ] outputs/eval/ragas_scores.json added
[ ] Screenshot of Streamlit UI added

COMMIT MESSAGE:
[Day-15] Enterprise Knowledge Navigator - Complete Advanced RAG System

EMAIL TO: ceo@algoprofessor.com
Subject: Day 15 Milestone - Enterprise Knowledge Navigator Submitted


## PROJECT COMPLETE!

### Full system built across 4 days:

**Day 1:** Document ingestion pipeline
- 50+ Tesla documents collected
- PDF and text loader with metadata
- Intelligent chunking with overlap
- ChromaDB vector store + BM25 keyword index

**Day 2:** Advanced retrieval pipeline
- HyDE hypothetical document embeddings
- Multi-query variant generation
- Hybrid search with RRF fusion
- Cross-encoder reranking
- Conversational memory with compression

**Day 3:** Graph RAG and backend
- Neo4j knowledge graph
- Entity extraction from documents
- Graph-enhanced search
- FastAPI REST backend

**Day 4:** Frontend and evaluation
- Streamlit chat interface
- Source highlighting with page numbers
- RAGAS evaluation dashboard
- GitHub README and submission


